In [13]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from arch import arch_model
from data_wrangling import *

## Pulling data from various data files

In [2]:
# data_dxy = yf.download("DX-Y.NYB", start="2000-01-01", end="2024-01-01", threads=False)
# data_2yr = yf.download("^IRX", start="2000-01-01", end="2024-01-01", threads=False)
# data_10yr = yf.download("^TNX", start="2000-01-01", end="2024-01-01", threads=False)
# data_vix = yf.download("^VIX", start="2000-01-01", end="2024-01-01", threads=False)

data_dxy = pd.read_csv("data/data_dxy.csv", index_col=0).Close
data_dxy.index = pd.to_datetime(data_dxy.index)
data_2yr = pd.read_csv("data/data_2yr.csv", index_col=0).Close
data_10yr = pd.read_csv("data/data_10yr.csv", index_col=0).Close
data_vix = pd.read_csv("data/data_vix.csv", index_col=0).Close
data_vix.index = pd.to_datetime(data_vix.index)

data_2s10s = data_2yr - data_10yr
data_2s10s.index = pd.to_datetime(data_2s10s.index)

aapl_time_series = pd.read_hdf("data/all_tickers_time_series.hf5", key="AAPL").drop_duplicates().set_index("date")
aapl_returns = aapl_time_series.prc.pct_change()

In [3]:
print(len(aapl_time_series.index))
print(len(data_dxy.index))
print(len(data_2s10s.index))
print(len(data_vix.index))

6037
6064
6031
6037


Apparantly these dataframes have a difference in data lengths

In [4]:
dates_aapl = set(aapl_time_series.index)
dates_dxy = set(data_dxy.index)
dates_2s10s = set(data_2s10s.index)
dates_vix = set(data_vix.index)

common_dates = dates_aapl & dates_dxy & dates_2s10s & dates_vix
not_common_dates = (dates_aapl | dates_dxy | dates_2s10s | dates_vix) - common_dates

print(len(common_dates))
print(len(not_common_dates))

6030
36


In [5]:
sorted(list(not_common_dates))

[Timestamp('2000-01-17 00:00:00'),
 Timestamp('2000-02-21 00:00:00'),
 Timestamp('2000-07-04 00:00:00'),
 Timestamp('2000-11-23 00:00:00'),
 Timestamp('2001-01-15 00:00:00'),
 Timestamp('2001-02-19 00:00:00'),
 Timestamp('2001-07-04 00:00:00'),
 Timestamp('2001-09-03 00:00:00'),
 Timestamp('2001-11-22 00:00:00'),
 Timestamp('2002-01-21 00:00:00'),
 Timestamp('2002-02-18 00:00:00'),
 Timestamp('2002-05-27 00:00:00'),
 Timestamp('2002-07-04 00:00:00'),
 Timestamp('2002-11-28 00:00:00'),
 Timestamp('2003-07-04 00:00:00'),
 Timestamp('2003-09-01 00:00:00'),
 Timestamp('2003-11-11 00:00:00'),
 Timestamp('2004-01-19 00:00:00'),
 Timestamp('2004-02-16 00:00:00'),
 Timestamp('2004-06-11 00:00:00'),
 Timestamp('2004-07-05 00:00:00'),
 Timestamp('2004-09-06 00:00:00'),
 Timestamp('2004-11-25 00:00:00'),
 Timestamp('2004-12-24 00:00:00'),
 Timestamp('2005-01-17 00:00:00'),
 Timestamp('2005-02-21 00:00:00'),
 Timestamp('2005-07-04 00:00:00'),
 Timestamp('2005-10-10 00:00:00'),
 Timestamp('2005-11-

Turns out these dates are the days where it is a banking holiday, thus stock market not opened, but various indexes continues to count <br>
Note, 2016-10-10 and 2016-11-11 are banking holidays, but AAPL continued to trade

In [6]:
new_aapl = aapl_time_series[aapl_time_series.index.isin(common_dates)]
new_data_dxy = data_dxy[data_dxy.index.isin(common_dates)]
new_data_2s10s = data_2s10s[data_2s10s.index.isin(common_dates)]
new_data_vix = data_vix[data_vix.index.isin(common_dates)]

## EDA

In [7]:
adf_test(new_data_dxy)
print('---')
adf_test(new_data_2s10s)
print('---')
adf_test(new_data_vix)

ADF Statistic: -1.5625279264735055
p-value: 0.5023845733881521
Non-stationary: Consider differencing or other transformations
---
ADF Statistic: -1.4974466159694706
p-value: 0.5347740811956415
Non-stationary: Consider differencing or other transformations
---
ADF Statistic: -5.809785756316987
p-value: 4.429021238909754e-07
Stationary: No differencing required


## Testing DXY

In [8]:
goldfeld_quandt_test(new_aapl['prc'].pct_change().dropna(), new_data_dxy.pct_change().dropna())

{'F-statistic': 0.6113312480717363, 'p-value': 0.9999999999999999}

Based on GQ test, since p-value is close to 1, we do not reject H0, suggesting that homoscedasticity, meeting the assumption of constant variance

In [9]:
white_test(new_aapl['prc'].pct_change().dropna(), new_data_dxy.pct_change().dropna())

{'LM-statistic': 1.3307228729956266,
 'LM-test p-value': 0.514087684071219,
 'F-statistic': 0.6651771741276042,
 'F-test p-value': 0.5142201696270757}

Based on White's test, failed to reject H0 as both LM-statistic and F-statistics are both not significant. This suggests that variance of residual is homoscedastic

## Testing 2s10s

In [22]:
aapl_returns = new_aapl['prc'].pct_change().dropna()
dxy_returns = new_data_2s10s.pct_change().dropna()

merged_data = pd.merge(aapl_returns, dxy_returns, left_index=True, right_index=True, how='inner')

merged_data = merged_data.replace([np.inf, -np.inf], np.nan).dropna()

gq_test_results = goldfeld_quandt_test(merged_data.iloc[:, 0], merged_data.iloc[:, 1])
white_test_results = white_test(merged_data.iloc[:, 0], merged_data.iloc[:, 1])

print(gq_test_results)
print(white_test_results)

{'F-statistic': 3.4617351398397833, 'p-value': 4.929946278060511e-240}
{'LM-statistic': 0.2703809463549116, 'LM-test p-value': 0.8735495080903555, 'F-statistic': 0.13512925303093257, 'F-test p-value': 0.8736056361016313}


GQ test and White's test showing conflicting results (GQ suggesting heteroscedasticity while White's showing homoscedasticity). White's test generally considered stricter and more comprehensive as it makes few assumptions of form of heteroscedasticity, thus we will follow White's results

## Testing VIX

In [23]:
goldfeld_quandt_test(new_aapl['prc'].pct_change().dropna(), new_data_vix.pct_change().dropna())

{'F-statistic': 1.632306755824517, 'p-value': 3.974481603250593e-41}

In [24]:
white_test(new_aapl['prc'].pct_change().dropna(), new_data_vix.pct_change().dropna())

{'LM-statistic': 746.6320908035916,
 'LM-test p-value': 7.42850595868739e-163,
 'F-statistic': 425.87008861589266,
 'F-test p-value': 1.0084858502142215e-173}

Both GQ and White suggests presence of heteroscedasticity by rejecting null hypothesis of homoscedasticity.